In [ ]:
pip install pandas nltk rank-bm25 sentence-transformers scikit-learn ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 95.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

In [ ]:
pip install --upgrade gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 6.1 MB/s eta 0:00:00


In [ ]:
# Import all required libraries
import json
import re
import numpy as np
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from IPython.display import HTML
from collections import defaultdict
import operator
import math

In [ ]:
# 1. Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive',force_remount= True)

# 2. Set the path to your uploaded file
file_path = '/content/drive/MyDrive/Colab Notebooks/info.r/arxiv-metadata-oai-snapshot.json'  # Use the correct file path


# =========================
#  Retrieval Functions
# =========================

# Boolean Retrieval Function
def boolean_retrieval(query, inverted_index, total_docs):
    """Process Boolean queries (AND/OR/NOT) using the inverted index"""
    # Split query into tokens (simple splitting; no nested parentheses)
    tokens = query.split()
    cleaned_terms = [process_query(token) for token in tokens if token not in {'AND', 'OR', 'NOT'}]
    operators = [token for token in tokens if token in {'AND', 'OR', 'NOT'}]

    # Get postings lists for terms
    postings = {}
    for term in cleaned_terms:
        postings[term] = set(inverted_index.get(term, {}).keys())

    # Process Boolean operations (left-to-right evaluation)
    if not postings:
        return []

    result = postings[cleaned_terms[0]]
    for i, op in enumerate(operators):
        next_term = cleaned_terms[i+1]
        next_set = postings.get(next_term, set())

        if op == 'AND':
            result = result.intersection(next_set)
        elif op == 'OR':
            result = result.union(next_set)
        elif op == 'NOT':
            result = result.difference(next_set)

    return list(result)

# TF-IDF Retrieval Function
def tfidf_retrieval(query, inverted_index, doc_freq, total_docs, top_n=100):
    """Rank documents using TF-IDF scores"""
    cleaned_query = process_query(query)
    query_terms = cleaned_query.split()

    # Calculate IDF for each query term
    idf = {term: math.log(total_docs / (doc_freq.get(term, 0) + 1e-10)) for term in query_terms}

    # Calculate TF-IDF scores for documents
    doc_scores = defaultdict(float)
    for term in query_terms:
        if term not in inverted_index:
            continue
        for doc_id, tf in inverted_index[term].items():
            doc_scores[doc_id] += tf * idf[term]

    # Sort documents by score
    sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
    top_indices = [doc_id for doc_id, _ in sorted_docs[:top_n]]
    return top_indices

# BM25 Retrieval Function
def bm25_retrieval(query, bm25_index, top_n=100):
    """First-stage retrieval using BM25"""
    tokenized_query = query.split()
    doc_scores = bm25_index.get_scores(tokenized_query)
    top_indices = np.argsort(doc_scores)[-top_n:][::-1]
    return top_indices, doc_scores[top_indices]

# SBERT Re-ranking Function
def sbert_reranking(query, doc_indices, sbert_model, doc_embeddings, top_k=100):
    """Second-stage semantic re-ranking"""
    query_embedding = sbert_model.encode(query)
    doc_embeddings_subset = doc_embeddings[doc_indices]

    similarities = np.dot(doc_embeddings_subset, query_embedding) / (
        np.linalg.norm(doc_embeddings_subset, axis=1) * np.linalg.norm(query_embedding))

    top_k_idx = np.argsort(similarities)[-top_k:][::-1]
    return doc_indices[top_k_idx], similarities[top_k_idx]


# =========================
#  Display & Highlight Functions
# =========================

# Highlighting Function
def highlight_terms(text, query_terms):
    """Highlight query terms in text using HTML bold tags"""
    tokens = word_tokenize(text)
    highlighted = []
    for token in tokens:
        cleaned_token = clean_text(token)
        if cleaned_token in query_terms:
            highlighted.append(f"<b>{token}</b>")
        else:
            highlighted.append(token)
    return ' '.join(highlighted)


def display_results(doc_indices, scores, df, query_terms, index_to_arxiv=None):
    """Display results in a formatted table"""
    results = []
    for idx, score in zip(doc_indices, scores):
        paper = df.iloc[idx]
        highlighted_title = highlight_terms(paper['title'], query_terms)
        highlighted_abstract = highlight_terms(paper['abstract'], query_terms)
        # Use mapping to fetch the original arXiv ID if provided
        doc_id = index_to_arxiv.get(idx, idx) if index_to_arxiv else idx
        results.append({
            'DocID': doc_id,
            'Title': highlighted_title,
            #'Title': paper['title'],
            'Authors': paper['authors_combined'],
            'Year': paper['year'],
            'Categories': ', '.join(paper['categories']),
            'Score': f"{score:.4f}",
            'Abstract': highlighted_abstract[:150] + '...'
            #'Abstract': paper['abstract'][:150] + '...'
        })
    return pd.DataFrame(results)


# =========================
#  Data Loading and Preprocessing
# =========================

# Download NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('stopwords')

# Initialize NLP tools
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# 1. Data Loading
def load_arxiv_data(filepath, limit=10000):
    """Load arXiv dataset from JSON file (line-delimited)"""
    data = []
    with open(filepath, 'r') as f:
        for i, line in enumerate(f):
            if limit and i >= limit:
                break
            item = json.loads(line)
            data.append({
                "id": item.get("id", ""),
                "title": item.get("title", ""),
                "abstract": item.get("abstract", ""),
                "authors": item.get("authors", ""),
                "categories": item.get("categories", ""),
                "update_date": item.get("update_date", "")
            })
    return pd.DataFrame(data)

# 2. Text Preprocessing
def clean_text(text):
    """Clean and preprocess text"""
    if not isinstance(text, str):
        return ""

    # Lowercase
    text = text.lower()

    # Remove special characters and numbers (keep basic punctuation)
    text = re.sub(r'[^a-zA-Z\s.,;!?]', '', text)

    # Tokenize
    tokens = word_tokenize(text)

    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(token)
             for token in tokens
             if token not in stop_words and len(token) > 1]

    return ' '.join(tokens)

def preprocess_papers(df):
    """Preprocess the entire dataframe"""
    # Handle missing values
    df.fillna({'title': '', 'abstract': '', 'authors': '', 'categories': ''}, inplace=True)

    # Combine relevant fields
    df['full_text'] = df['title'] + ' ' + df['abstract']

    # Clean text fields
    print("Cleaning text data...")
    df['cleaned_title'] = df['title'].apply(clean_text)
    df['cleaned_abstract'] = df['abstract'].apply(clean_text)
    df['cleaned_text'] = df['full_text'].apply(clean_text)

    # Process authors
    df['authors_combined'] = df['authors'].astype(str)

    # Process categories
    df['categories'] = df['categories'].apply(
        lambda x: [c.split('.')[0] for c in x.split()] if isinstance(x, str) else [])

    # Extract year
    df['year'] = pd.to_datetime(df['update_date']).dt.year

    return df


# =========================
#  Index Creation
# =========================
# 3. Index Creation
def create_indices(df):
    """Create all necessary search indices"""
    print("Creating BM25 index...")
    bm25_index = BM25Okapi([doc.split() for doc in df['cleaned_text']])

    print("Creating Inverted Index...")
    inverted_index = defaultdict(dict)  # {term: {doc_id: term_frequency}}
    doc_freq = defaultdict(int)         # {term: number of documents containing the term}
    for doc_id, text in enumerate(df['cleaned_text']):
        terms = text.split()
        term_counts = defaultdict(int)
        # Count term frequencies in the current document
        for term in terms:
            term_counts[term] += 1
        # Update inverted index and document frequency
        for term, count in term_counts.items():
            inverted_index[term][doc_id] = count
            doc_freq[term] += 1  # Increment document count for the term

    print("Creating SBERT embeddings...")
    sbert_model = SentenceTransformer('all-mpnet-base-v2')
    doc_embeddings = sbert_model.encode(df['cleaned_text'].tolist())

    return bm25_index, sbert_model, doc_embeddings, inverted_index, doc_freq


# =========================
#  Search Pipeline
# =========================
# 4. Search Pipeline
def process_query(query):
    """Process user query with the same cleaning as documents"""
    return clean_text(query)


# Modified search_papers to optionally return ranked document indices and scores for evaluation
def search_papers(query, bm25_index, sbert_model, doc_embeddings, df, inverted_index, doc_freq, method='hybrid', return_ranked=False, index_to_arxiv=None):
    """Search with Boolean, TF-IDF, or hybrid BM25+SBERT"""
    cleaned_query = process_query(query)
    query_terms = cleaned_query.split()

    if method == 'boolean':
        # Boolean Retrieval
        total_docs = len(df)
        doc_indices = boolean_retrieval(query, inverted_index, total_docs)
        scores = [1.0] * len(doc_indices)  # Placeholder score
    elif method == 'tfidf':
        # TF-IDF Retrieval
        total_docs = len(df)
        doc_indices = tfidf_retrieval(query, inverted_index, doc_freq, total_docs)
        scores = [1.0] * len(doc_indices)
    else:
        # Default: BM25 + SBERT
        bm25_doc_indices, _ = bm25_retrieval(cleaned_query, bm25_index)
        #final_indices, final_scores = sbert_reranking(cleaned_query, bm25_doc_indices, sbert_model, doc_embeddings)
        doc_indices, scores = sbert_reranking(cleaned_query, bm25_doc_indices, sbert_model, doc_embeddings)
        #doc_indices, scores = final_indices, final_scores
    if return_ranked:
        return doc_indices, scores
    else:
        return display_results(doc_indices, scores, df, query_terms, index_to_arxiv=index_to_arxiv)
    # Display with highlighting
    #return display_results(doc_indices, scores, df, query_terms)


# =========================
#  Evaluation Metric Functions
# =========================
def precision_at_k(retrieved, relevant):
    """Compute precision at k given retrieved and relevant doc IDs."""
    retrieved_set = set(retrieved)
    if not retrieved:
        return 0.0
    return len(retrieved_set & set(relevant)) / len(retrieved)

def recall_at_k(retrieved, relevant):
    """Compute recall at k."""
    if not relevant:
        return 0.0
    return len(set(retrieved) & set(relevant)) / len(relevant)

def f1_score(precision, recall):
    """Compute F1 score from precision and recall values."""
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def average_precision(retrieved, relevant):
    """Compute the average precision for a single query."""
    if not relevant:
        return 0.0
    ap = 0.0
    hit_count = 0.0
    for i, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant:
            hit_count += 1
            ap += hit_count / i
    return ap / len(relevant)

def ndcg_at_k(retrieved, relevant, k):
    """Compute nDCG at k. Here binary relevance (1 if relevant, 0 if not)."""
    dcg = 0.0
    for i, doc_id in enumerate(retrieved[:k], start=1):
        rel = 1 if doc_id in relevant else 0
        dcg += (2**rel - 1) / math.log2(i + 1)
    # Compute ideal DCG
    ideal_rel = [1] * min(len(relevant), k)
    idcg = sum((2**rel - 1) / math.log2(i + 1) for i, rel in enumerate(ideal_rel, start=1))
    if idcg == 0:
        return 0.0
    return dcg / idcg

def reciprocal_rank(retrieved, relevant):
    """Compute the reciprocal rank for a single query."""
    for i, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant:
            return 1.0 / i
    return 0.0


def evaluate_ir_system(queries, ground_truth, bm25_index, sbert_model, doc_embeddings, df, inverted_index, doc_freq, method='hybrid', k=100):
    """Evaluate the IR system over a list of queries with provided ground truth"""
    precision_list = []
    recall_list = []
    f1_list = []
    ap_list = []
    ndcg_list = []
    rr_list = []

    for query in queries:
        relevant = set(ground_truth.get(query, []))
        # Get ranked list of document IDs using the search pipeline
        retrieved, _ = search_papers(query, bm25_index, sbert_model, doc_embeddings, df, inverted_index, doc_freq, method, return_ranked=True)
        retrieved = list(retrieved)[:k]  # consider top k

        prec = precision_at_k(retrieved, relevant)
        rec = recall_at_k(retrieved, relevant)
        f1 = f1_score(prec, rec)
        ap = average_precision(retrieved, relevant)
        ndcg = ndcg_at_k(retrieved, relevant, k)
        rr = reciprocal_rank(retrieved, relevant)

        precision_list.append(prec)
        recall_list.append(rec)
        f1_list.append(f1)
        ap_list.append(ap)
        ndcg_list.append(ndcg)
        rr_list.append(rr)

        print(f"Query: '{query}'")
        print(f"  Precision@{k}: {prec:.4f}")
        print(f"  Recall@{k}:    {rec:.4f}")
        print(f"  F1 Score:      {f1:.4f}")
        print(f"  Average Precision: {ap:.4f}")
        print(f"  nDCG@{k}:      {ndcg:.4f}")
        print(f"  Reciprocal Rank: {rr:.4f}")
        print("-" * 40)

    metrics = {
        "Average Precision": np.mean(precision_list),
        "Average Recall": np.mean(recall_list),
        "Average F1": np.mean(f1_list),
        "MAP": np.mean(ap_list),
        "Mean nDCG": np.mean(ndcg_list),
        "MRR": np.mean(rr_list)
    }

    print("\nOverall Evaluation Metrics:")
    for key, value in metrics.items():
        print(f"{key}: {value:.4f}")

    return metrics


if __name__ == "__main__":
    print("Loading data...")
    df = load_arxiv_data(file_path, limit=10000)
    df = preprocess_papers(df)

    # Create mapping: DataFrame row index -> original arXiv ID
    index_to_arxiv = df['id'].to_dict()

    # Create all indices
    bm25_index, sbert_model, doc_embeddings, inverted_index, doc_freq = create_indices(df)
'''
    # Get user input
    query = input("Enter your search query: ")
    method = input("Choose method (hybrid/boolean/tfidf): ").strip().lower()

    # Perform search
    results = search_papers(
        query,
        bm25_index,
        sbert_model,
        doc_embeddings,
        df,
        inverted_index,
        doc_freq,
        method=method,
        return_ranked=False,
        index_to_arxiv=index_to_arxiv
    )

    # Display results
    display(HTML(results.to_html(escape=False)))
'''

Mounted at /content/drive


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Loading data...
Cleaning text data...
Creating BM25 index...
Creating Inverted Index...
Creating SBERT embeddings...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

'\n    # Get user input\n    query = input("Enter your search query: ")\n    method = input("Choose method (hybrid/boolean/tfidf): ").strip().lower()\n\n    # Perform search\n    results = search_papers(\n        query,\n        bm25_index,\n        sbert_model,\n        doc_embeddings,\n        df,\n        inverted_index,\n        doc_freq,\n        method=method,\n        return_ranked=False,\n        index_to_arxiv=index_to_arxiv\n    )\n\n    # Display results\n    display(HTML(results.to_html(escape=False)))\n'

In [ ]:
import gradio as gr
from IPython.display import display

# Define a function that calls your search engine.
def search_interface(query, method):
    # Simple error checking: if the query is empty, return a message.
    if not query.strip():
        return "<p>Please enter a search query.</p>"

    # Call your existing search_papers function with the provided parameters.
    # It must return a DataFrame with the desired columns (and formatted via display_results).
    results = search_papers(
        query,
        bm25_index,
        sbert_model,
        doc_embeddings,
        df,
        inverted_index,
        doc_freq,
        method=method,
        return_ranked=False,
        index_to_arxiv=index_to_arxiv
    )

    # Convert the DataFrame to an HTML table. (escape=False so that highlighted text renders as HTML)
    return results.to_html(escape=False)

# Build a Gradio Blocks interface with a custom layout.
with gr.Blocks() as demo:
    # Create a header row with your images.
    with gr.Row():
        # Replace with image link
        img3 = gr.Image(value="/content/drive/MyDrive/Colab Notebooks/info.r/SE_Logo.png", label="", show_label=False, interactive=False)

    # You can add a title or description if you like
    gr.Markdown("## IR Search Engine GUI")

    # Create an input row for query and method.
    with gr.Row():
        query_input = gr.Textbox(label="Search Query", placeholder="Type your search query here...", lines=1)
        method_dropdown = gr.Dropdown(
            label="Retrieval Method",
            choices=["hybrid", "boolean", "tfidf"],
            value="hybrid"
        )
        search_button = gr.Button("Search")

    # Output area that will display the HTML table with results.
    output_html = gr.HTML(label="Search Results")

    # When the search button is clicked, call the search_interface function.
    search_button.click(fn=search_interface, inputs=[query_input, method_dropdown], outputs=output_html)

# Launch the Gradio app. In Colab, this will create a temporary public URL.
demo.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9a7d4ad7ca391b315a.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
    # ---------------------
    # Evaluation
    # ---------------------
    # For evaluation, we need a set of test queries and associated ground truth
    # The ground_truth dictionary should map each query to a set (or list) of relevant document indices

    # Here is an example:
    test_queries = [
        "deep learning papers",
    ]
    # find relevance papers manually:
    ground_truth = {
        "deep learning papers": {0,1,2,4,5,6,7,8,9,11,12,13,16,20,25,28,33,46,51,53,61,86,88},
    }

    print("\nEvaluating IR system on test queries...")
    evaluate_ir_system(test_queries, ground_truth, bm25_index, sbert_model, doc_embeddings, df, inverted_index, doc_freq, method='hybrid', k=100)


Evaluating IR system on test queries...
Query: 'deep learning papers'
  Precision@100: 0.0100
  Recall@100:    0.0435
  F1 Score:      0.0163
  Average Precision: 0.0009
  nDCG@100:      0.0232
  Reciprocal Rank: 0.0213
----------------------------------------

Overall Evaluation Metrics:
Average Precision: 0.0100
Average Recall: 0.0435
Average F1: 0.0163
MAP: 0.0009
Mean nDCG: 0.0232
MRR: 0.0213


{'Average Precision': np.float64(0.01),
 'Average Recall': np.float64(0.043478260869565216),
 'Average F1': np.float64(0.016260162601626018),
 'MAP': np.float64(0.0009250693802035152),
 'Mean nDCG': np.float64(0.023242424817620892),
 'MRR': np.float64(0.02127659574468085)}